In [ ]:
!apt-get update -y
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama requests

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,917 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,622 kB]
Get:13 http://security.ubuntu.com/ubuntu jam

In [ ]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(5)

print("Ollama server started.")

Ollama server started.


In [ ]:
!ollama pull qwen2.5-coder:7b

In [ ]:
import ollama

try:
    response = ollama.chat(
        model="qwen2.5-coder:7b",
        messages=[{"role": "user", "content": "Why is the sky blue?"}]
    )
    print(response["message"]["content"])
except Exception as e:
    print(f"Test failed: {e}")

The sky appears blue due to a phenomenon called Rayleigh scattering. This occurs when sunlight enters Earth's atmosphere and interacts with gas molecules and tiny particles like dust. The shorter wavelengths of light, such as blue and violet, are scattered more easily than longer wavelengths like red or orange. 

When sunlight passes through the atmosphere, these shorter wavelength blues get dispersed in all directions, making it appear that the sky is blue to our eyes. However, during sunrise and sunset, when the sun is closer to the horizon, the sky can appear more pink or orange because the light has to pass through a thicker layer of atmosphere, causing even more scattering.

This same principle explains why sunsets are often red or orange: longer wavelengths are scattered less and reach our eyes after passing through a significant portion of Earth's atmosphere.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
import requests
import json
from pathlib import Path

URL = "http://localhost:11434/api/chat"
MODEL = "qwen2.5-coder:7b"

SCHEMA = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "confirmed_errors": {
            "type": "array",
            "items": {"type": "string"}
        },
        "warnings": {
            "type": "array",
            "items": {"type": "string"}
        },
        "security_events": {
            "type": "array",
            "items": {"type": "string"}
        },
        "possible_causes": {
            "type": "array",
            "items": {"type": "string"}
        },
        "recommended_next_steps": {
            "type": "array",
            "items": {"type": "string"}
        }
    },
    "required": [
        "summary",
        "confirmed_errors",
        "warnings",
        "security_events",
        "possible_causes",
        "recommended_next_steps"
    ]
}

LOG_DIR = Path("/content/drive/MyDrive/logs")

In [ ]:
def ask_ollama(prompt: str) -> str:
    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a Linux log analysis assistant. "
                    "Return concise, accurate findings only. "
                    "Do not invent facts. "
                    "Do not copy raw log lines into possible_causes unless explicitly asked."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            }
        ],
        "stream": False,
        "format": SCHEMA,
        "options": {
            "temperature": 0
        }
    }

    response = requests.post(URL, json=payload, timeout=600) # Increased timeout to 600 seconds (10 minutes)
    response.raise_for_status()

    data = response.json()
    return data["message"]["content"]


def build_prompt(file_name: str, log_text: str) -> str:
    return f"""
Analyze this Linux log file: {file_name}

Tasks:
1. Summarize the main problems.
2. List confirmed errors from most critical to least critical.
3. List warnings.
4. List security-relevant events.
5. List possible high-level causes.
6. List recommended next steps.

Rules:
- Use only evidence explicitly present in the log.
- Do not invent facts.
- Do not say the system is healthy or started successfully unless the log explicitly says so.
- Do not treat normal CRON activity, normal systemd start/finish messages, or successful logins as errors.
- Treat failed SSH logins as security-relevant.
- Treat OOM kills, filesystem errors, repeated HTTP 500 errors, PostgreSQL connection errors, and Docker/container failures as important when present.
- possible_causes must be short explanations, not copied log lines.
- If a section has no items, return an empty list.
- Keep the response concise.

Log contents:
{log_text}
""".strip()


def normalize_list(items) -> list[str]:
    if not isinstance(items, list):
        return []

    normalized = []
    for item in items:
        if isinstance(item, str):
            cleaned = item.strip()
            if cleaned and cleaned.lower() != "string":
                normalized.append(cleaned)
        elif isinstance(item, dict):
            parts = [str(value).strip() for value in item.values() if str(value).strip()]
            if parts:
                joined = " | ".join(parts)
                if joined.lower() != "string":
                    normalized.append(joined)
        else:
            cleaned = str(item).strip()
            if cleaned and cleaned.lower() != "string":
                normalized.append(cleaned)

    return normalized


# def dedupe_list(items: list[str]) -> list[str]:
#     seen = set()
#     deduped = []

#     for item in items:
#         key = item.strip().lower()
#         if key and key not in seen:
#             seen.add(key)
#             deduped.append(item)

#     return deduped


def clean_possible_causes(items: list[str]) -> list[str]:
    cleaned = []

    for item in items:
        lower_item = item.lower().strip()

        if lower_item == "string":
            continue

        if any(token in lower_item for token in [
            "cron[",
            "systemd[",
            "sshd[",
            "nginx[",
            "postgres[",
            "docker[",
            "kernel:",
            "cmd (",
            "accepted password",
            "connection closed by authenticating user"
        ]):
            continue

        cleaned.append(item)

    if not cleaned:
        return ["Unclear from the log"]

    return cleaned


def normalize_result(result: dict) -> dict:
    summary = str(result.get("summary", "No summary provided.")).strip()
    if summary.lower() == "string":
        summary = "Model returned placeholder text instead of real analysis."

    normalized = {
        "summary": summary,
        # "confirmed_errors": dedupe_list(normalize_list(result.get("confirmed_errors", []))),
        # "warnings": dedupe_list(normalize_list(result.get("warnings", []))),
        # "security_events": dedupe_list(normalize_list(result.get("security_events", []))),
        # "possible_causes": clean_possible_causes(
        #     dedupe_list(normalize_list(result.get("possible_causes", [])))
        # ),
        # "recommended_next_steps": dedupe_list(
        #     normalize_list(result.get("recommended_next_steps", []))
        # ),
    }

    return normalized


def print_section(title: str, items: list[str]) -> None:
    print(f"\n{title}:")
    if items:
        for i, item in enumerate(items, start=1):
            print(f"{i}. {item}")
    else:
        print("None")


def print_report(result: dict) -> None:
    print("\n=== Analysis Result ===\n")

    print("Summary:")
    print(result["summary"] if result["summary"] else "No summary provided.")

    # print_section("Confirmed Errors", result["confirmed_errors"])
    # print_section("Warnings", result["warnings"])
    # print_section("Security Events", result["security_events"])
    # print_section("Possible Causes", result["possible_causes"])
    # print_section("Recommended Next Steps", result["recommended_next_steps"])


def choose_log_file(files: list[Path]) -> Path:
    for i, file in enumerate(files, start=1):
        print(f"{i}. {file.name}")

    while True:
        choice = input("Which log file do you want to analyze? ").strip()

        if not choice.isdigit():
            print("Please enter a valid number.")
            continue

        choice_num = int(choice)

        if 1 <= choice_num <= len(files):
            return files[choice_num - 1]

        print("That number is not in the list.")

In [ ]:
!mkdir -p /content/drive/MyDrive/logs

In [ ]:
def main() -> None:
    if not LOG_DIR.exists():
        print(f"Error: '{LOG_DIR}' folder does not exist.")
        return

    files = [file for file in LOG_DIR.iterdir() if file.is_file()]

    if not files:
        print(f"No log files found in '{LOG_DIR}'.")
        return

    selected_file = choose_log_file(files)
    log_text = selected_file.read_text(encoding="utf-8", errors="replace")

    max_chars = 4000  # Reduced from 12000 to shorten prompt and avoid timeout
    if len(log_text) > max_chars:
        log_text = log_text[:max_chars]

    prompt = build_prompt(selected_file.name, log_text)

    print("\nAnalyzing log file...\n")
    reply = ask_ollama(prompt)

    print("=== Raw Model Reply ===")
    print(reply)
    print("=== End Raw Model Reply ===")

    try:
        result = json.loads(reply)
        normalized_result = normalize_result(result)
        print_report(normalized_result)
    except json.JSONDecodeError:
        print("\nModel did not return valid JSON.")
        print("Raw response above could not be parsed.")


if __name__ == "__main__":
    main()

1. test.log
Which log file do you want to analyze? 1

Analyzing log file...

=== Raw Model Reply ===
{ "summary": "The log indicates several critical issues including an out-of-memory error, repeated HTTP 500 errors, a PostgreSQL connection loss, and a Docker container failure. There are also multiple failed SSH login attempts for invalid users.", "confirmed_errors": [ "Out of memory: Kill process 2145 (python3) score 987 or sacrifice child", "500 Internal Server Error: /api/login", "connection to client lost", "failed to start container" ], "warnings": [], "security_events": [ "Failed password for invalid user admin from 203.0.113.12 port 49821 ssh2", "Failed password for user root from 198.51.100.23 port 60022 ssh2" ], "possible_causes": [ "Insufficient memory allocated to the system", "Application errors in the web server or database", "Misconfigured Docker container settings", "Brute force SSH login attempts" ], "recommended_next_steps": [ "Investigate and resolve the memory issue 